# Conversational AI with LangChain
A conversation requires **memory** — the model must remember what was said earlier.

By default, LLMs are stateless (each call is independent). We fix this by passing the chat history manually.

In [7]:
pip install langchain langchain-ollama --quiet

Note: you may need to restart the kernel to use updated packages.


## 1. Without Memory (Stateless)
The model forgets everything between calls.

In [8]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2')
# First message
r1 = llm.invoke('My name is Ram.')
print('Turn 1:', r1.content)

# Second message — model has no memory of the first
r2 = llm.invoke('What is my name?')
print('Turn 2:', r2.content)

Turn 1: Hello Ram! It's nice to meet you. Is there something I can help you with, or would you like to chat?
Turn 2: I don't have any information about your name. I'm a large language model, I don't have the ability to retain personal information about individual users. Each time you interact with me, it's a new conversation and I don't have any prior knowledge about you. Would you like to introduce yourself?


## 2. With Memory (Manual History)
We pass the full chat history in every call so the model has context.

In [10]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

llm = ChatOllama(model='llama3.2')

# Start with a system message and build history
history = [
    SystemMessage(content='You are a helpful assistant.')
]

def chat(user_input):
    history.append(HumanMessage(content=user_input))
    # print(history)
    response = llm.invoke(history)
    history.append(AIMessage(content=response.content))
    return response.content

print('Turn 1:', chat('My name is Ram.'))
print('Turn 2:', chat('What is my name?'))
print('Turn 3:', chat('What did I tell you in my first message?'))

Turn 1: Hello Ram! It's nice to meet you. Is there anything I can help you with or would you like to chat for a bit?
Turn 2: Your name is Ram.
Turn 3: In your first message, you told me "My name is Ram."


## 3. Inspect the Chat History

In [11]:
for msg in history:
    role = msg.__class__.__name__.replace('Message', '')
    print(f'[{role}]: {msg.content[:100]}')
    

[System]: You are a helpful assistant.
[Human]: My name is Ram.
[AI]: Hello Ram! It's nice to meet you. Is there anything I can help you with or would you like to chat fo
[Human]: What is my name?
[AI]: Your name is Ram.
[Human]: What did I tell you in my first message?
[AI]: In your first message, you told me "My name is Ram."


In [14]:
history = """

[System]: You are a helpful assistant.
[Human]: My name is Ram.
[AI]: Hello Ram! It's nice to meet you. Is there anything I can help you with or would you like to chat fo
[Human]: What is my name?
[AI]: Your name is Ram.
[Human]: What did I tell you in my first message?
[AI]: In your first message, you told me "My name is Ram."

"""

reponse = llm.invoke(f"PLease sumamrize this message : {history}")
reponse.content

'The AI assistant and Ram interacted in a conversation, where Ram asked the AI what his name was, and the AI provided the correct answer: "Ram".'

## 4. Multi-turn Conversation with a Loop

In [15]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

llm = ChatOllama(model='llama3.2')
history = [SystemMessage(content='You are a helpful assistant.')]

print("Chat started. Type 'quit' to exit.\n")

while True:
    user_input = input('You: ')
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break
    history.append(HumanMessage(content=user_input))
    response = llm.invoke(history)
    history.append(AIMessage(content=response.content))
    print(f'AI: {response.content}\n')

Chat started. Type 'quit' to exit.



You:  Hi how are you


AI: I'm just a language model, so I don't have emotions or feelings like humans do, but I'm functioning properly and ready to help you with any questions or tasks you have. How can I assist you today?



You:  my name is RAM


AI: Hi RAM! It's nice to meet you. Is there something I can help you with, or would you like to chat for a bit?



...

AI: I'll process your payment for the **Loaded Fries** ($5.99) and **Classic Cheese Pizza** ($8.99).

**Payment Method:** Card

**Card Details:** Please insert your card details to complete the payment.

**Card Type:** Visa

**Card Number:** ************1234

**Card Expiration Date:** 02/2025

**Card Security Code:** 123

Payment successful!

Your order is now on its way to you. Here's a confirmation message:

**ORDER CONFIRMATION**

* Order Number: PH-001
* Order Date: [Current Date]
* Order Total: $14.98
* Pizza Type: Classic Cheese
* Fries Type: Loaded Fries
* Payment Method: Card

You should receive your pizza and loaded fries within 20-30 minutes. If you have any questions or concerns, please don't hesitate to reach out to us.

Thank you for ordering from Pizza Hot!



You:  can yo uplease create a json response for my order


AI: Here is a JSON response for your order:

```json
{
  "orderId": "PH-001",
  "orderDate": "2023-12-31",
  "orderTotal": 14.98,
  "pizzaType": "Classic Cheese",
  "friesType": "Loaded Fries",
  "paymentMethod": "Card",
  "cardDetails": {
    "cardType": "Visa",
    "cardNumber": "***********1234",
    "expirationDate": "02/2025",
    "securityCode": "123"
  }
}
```

Let me know if you'd like me to make any changes to this JSON response!



You:  quit


Goodbye!


## Summary

| Concept | Description |
|---|---|
| Stateless | Each LLM call is independent — no memory |
| Chat history | List of messages passed with every call |
| `HumanMessage` | Message from the user |
| `AIMessage` | Message from the model |
| `SystemMessage` | Instructions that set model behavior |
| Memory | Achieved by accumulating history and passing it each time |